In [ ]:
import json, re, sys, os, subprocess
from pathlib import Path
import pandas as pd

IN_COLAB = "google.colab" in sys.modules

# Where to clone from (private repos need a token in the URL)
REPO_URL = "https://github.com/BayJacon/Parliamentary-Data-Proj.git"
REPO_DIR = Path("/content/Parliamentary-Data-Proj")

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)   # get latest data
    os.chdir(REPO_DIR)

print("Running in:", "Colab" if IN_COLAB else "local")
print("Working folder:", Path.cwd())

Running in: Colab
Working folder: /content/Parliamentary-Data-Proj


In [9]:
def find_raw_dir(start=Path.cwd()):
    # Look in this folder and upwards, so it works if the notebook sits in a subfolder.
    for p in [start, *start.parents]:
        candidate = p / "hansard_output" / "raw_json"
        if candidate.exists():
            return candidate
    return None

RAW_DIR = find_raw_dir()
if RAW_DIR is None:
    raise FileNotFoundError(
        "No hansard_output/raw_json folder found. Run the downloader notebook first, "
        "or (in Colab) make sure the data was committed to the repo.")

files = sorted(RAW_DIR.glob("*.json"))
print("Data folder:", RAW_DIR.resolve())
print(len(files), "debate files")
for f in files[:5]:
    print("  ", f.name)

Data folder: /content/hansard_output/raw_json
5 debate files
   2026-10-27_50764AC6-99E1-4169-A4ED-335EB488A6D0.json
   2026-10-27_565BC6CF-D5F8-4AFB-9D06-1CBD965141BF.json
   2026-10-27_595A8127-E255-422E-AB60-30F8AD5D9C13.json
   2026-10-27_6CFF3D07-DAC3-4D00-B63E-FA6180E4BBAA.json
   2026-10-27_D51C150B-E116-40B8-B2AA-5160D86E1315.json


In [10]:
with open(files[0], encoding="utf-8") as fh:
    debate = json.load(fh)

print("Keys:", list(debate.keys()))
print("\nOverview:", debate["Overview"])
print("\nFirst two items:")
for item in debate["Items"][:2]:
    print(json.dumps(item, indent=2)[:800])

Keys: ['Overview', 'Navigator', 'Items', 'ChildDebates']

Overview: {'Id': 5428214, 'ExtId': '50764AC6-99E1-4169-A4ED-335EB488A6D0', 'Title': ' Services for Children in Care', 'HRSTag': 'hs_2cDebatedMotion', 'Date': '2026-10-27T00:00:00', 'Location': 'Commons Chamber', 'House': 'Commons', 'Source': 2, 'VolumeNo': 792, 'ContentLastUpdated': '2026-10-30T17:54:33.78', 'DebateTypeId': 1, 'SectionType': 1, 'NextDebateExtId': '', 'NextDebateTitle': '', 'PreviousDebateExtId': 'D51C150B-E116-40B8-B2AA-5160D86E1315', 'PreviousDebateTitle': 'Yorkhill Fire Station'}

First two items:
{
  "ItemType": "Contribution",
  "ItemId": 50844246,
  "MemberId": null,
  "AttributedTo": null,
  "Value": "<span id=\"272\" class=\"column-number column-only\" data-column-number=\"272\"></span>",
  "OrderInSection": 0,
  "Timecode": null,
  "ExternalId": null,
  "HRSTag": "hs_ColumnNumber",
  "HansardSection": null,
  "UIN": null,
  "IsReiteration": false
}
{
  "ItemType": "Contribution",
  "ItemId": 50844247,
  

In [12]:
frames = []
for f in files:
    with open(f, encoding="utf-8") as fh:
        d = json.load(fh)
    df = pd.json_normalize(d["Items"])
    df["debate_id"] = d["Overview"]["ExtId"]
    df["debate_title"] = d["Overview"]["Title"].strip()
    df["debate_date"] = d["Overview"]["Date"][:10]
    frames.append(df)

items = pd.concat(frames, ignore_index=True)
print("Shape:", items.shape)
print("Columns:", items.columns.tolist())
items.head(20)

Shape: (131, 15)
Columns: ['ItemType', 'ItemId', 'MemberId', 'AttributedTo', 'Value', 'OrderInSection', 'Timecode', 'ExternalId', 'HRSTag', 'HansardSection', 'UIN', 'IsReiteration', 'debate_id', 'debate_title', 'debate_date']


,ItemType,ItemId,MemberId,AttributedTo,Value,OrderInSection,Timecode,ExternalId,HRSTag,HansardSection,UIN,IsReiteration,debate_id,debate_title,debate_date
0,Contribution,50844246,NaN,None,"<span id=""272"" class=""column-number column-onl...",0,None,None,hs_ColumnNumber,None,None,False,50764AC6-99E1-4169-A4ED-335EB488A6D0,Services for Children in Care,2026-10-27
1,Contribution,50844247,NaN,None,"<em>Motion made, and Question proposed, </em>T...",2,None,26102760000537,hs_Para,CM-CP,None,False,50764AC6-99E1-4169-A4ED-335EB488A6D0,Services for Children in Care,2026-10-27
2,Timestamp,3706400,NaN,,18:10:00,3,None,None,None,CM-CP,None,False,50764AC6-99E1-4169-A4ED-335EB488A6D0,Services for Children in Care,2026-10-27
3,Contribution,50844248,5034.0,Josh Newbury (Cannock Chase) (Lab),"I would like to take this opportunity, as I ha...",4,2026-10-27T18:10:44,0EE6FA1B-2B93-4814-9CFA-B4CD284EB4AE,hs_Para,CM-CP,None,False,50764AC6-99E1-4169-A4ED-335EB488A6D0,Services for Children in Care,2026-10-27
4,Contribution,50844249,4131.0,Jim Shannon (Strangford) (DUP),I commend the hon. Gentleman for bringing forw...,5,None,C423E42A-DDC0-41D9-ABDB-C9E234D975A3,hs_Para,CM-CP,None,False,50764AC6-99E1-4169-A4ED-335EB488A6D0,Services for Children in Care,2026-10-27
5,Contribution,50844250,5034.0,Josh Newbury,I thank the hon. Gentleman for his kind words....,6,2026-10-27T16:59:14,B960348F-C69D-455D-9904-E9C145D57ABF,hs_Para,CM-CP,None,False,50764AC6-99E1-4169-A4ED-335EB488A6D0,Services for Children in Care,2026-10-27
6,Contribution,50844251,5334.0,Dr Allison Gardner (Stoke-on-Trent South) (Lab),I have constituents who are foster parents of ...,7,None,500D8612-63A3-4B3C-879B-BA1D6865B17B,hs_Para,CM-CP,None,False,50764AC6-99E1-4169-A4ED-335EB488A6D0,Services for Children in Care,2026-10-27
7,Contribution,50844252,5034.0,Josh Newbury,"I absolutely agree with my hon. Friend, who sp...",8,2026-10-27T18:23:19,F1BB73D8-66C1-4993-A7FF-7AC7F71BF75A,hs_Para,CM-CP,None,False,50764AC6-99E1-4169-A4ED-335EB488A6D0,Services for Children in Care,2026-10-27
8,Timestamp,3706401,NaN,,18:26:00,9,2026-10-27T18:24:59,None,None,CQ-CR,None,False,50764AC6-99E1-4169-A4ED-335EB488A6D0,Services for Children in Care,2026-10-27
9,Contribution,50844253,5071.0,The Parliamentary Under-Secretary of State for...,I thank Members for attending and contributing...,10,2026-10-27T18:26:23,CCA55FD8-A749-466F-B345-FD1A21239B12,hs_Para,CQ-CR,None,False,50764AC6-99E1-4169-A4ED-335EB488A6D0,Services for Children in Care,2026-10-27
